# Importation des Libriairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Affichage des graphiques dans le notebook
%matplotlib inline

# Paramètres d'affichage
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Charger les données

In [ ]:
df = pd.read_csv("../data/creditcard.csv")

In [ ]:
df.head()

# Comprendre la structure du dataset

In [ ]:
df.shape

# Informations générales

In [ ]:
df.info()

# Statistiques descriptives

In [ ]:
df.describe()

# Vérifier les valeurs manquantes

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum().sum()

# Vérifier les doublons

In [ ]:
df.duplicated().sum()
df = df.drop_duplicates()

# Étudier la variable cible

In [ ]:
df["isFraud"].value_counts()

In [ ]:
df["isFraud"].value_counts(normalize=True) * 100

# Visualiser les classes

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(x="isFraud", data=df)

plt.title("Distribution des transactions")
plt.show()

# Calcul du taux de fraude

In [ ]:
fraud_percentage = (
    df["isFraud"].value_counts(normalize=True)[1] * 100
)

print(f"Taux de fraude : {fraud_percentage:.4f}%")

# Analyse des types de transaction

In [ ]:
df["type"].value_counts()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    y="type",
    data=df,
    order=df["type"].value_counts().index
)

plt.title("Types de transaction")
plt.show()

# Fraudes par type de transaction

In [ ]:
fraud_by_type = pd.crosstab(
    df["type"],
    df["isFraud"]
)

fraud_by_type

In [ ]:
fraud_by_type.plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Fraudes par type de transaction")
plt.show()

# Taux de fraude par type

In [ ]:
fraud_rate = (
    df.groupby("type")["isFraud"]
      .mean()
      .sort_values(ascending=False)
      * 100
)

fraud_rate

# Analyse du montant

In [ ]:
df["amount"].describe()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    df["amount"],
    bins=100
)

plt.title("Distribution des montants")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    np.log1p(df["amount"]),
    bins=100
)

plt.title("Distribution logarithmique des montants")
plt.show()

# Comparer les montants selon la fraude

In [ ]:
sample_df = df.sample(
    100000,
    random_state=42
)

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="isFraud",
    y="amount",
    data=sample_df
)

plt.show()

# Etape du Feature Engineering

## Différence de solde côté émetteur

In [ ]:
df["balance_diff_orig"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

## Différence de solde côté destinataire

In [ ]:
df["balance_diff_dest"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

## Vérification

In [ ]:
df[
    [
        "oldbalanceOrg",
        "newbalanceOrig",
        "balance_diff_orig",
        "oldbalanceDest",
        "newbalanceDest",
        "balance_diff_dest"
    ]
].head()

# Transformer la variable type

In [ ]:
df["type"].unique()

## Encodage One-Hot

In [ ]:
df = pd.get_dummies(
    df,
    columns=["type"],
    drop_first=True
)

In [ ]:
df.head()

## Supprimer les identifiants

In [ ]:
df.drop(
    columns=["nameOrig", "nameDest"],
    inplace=True
)

In [ ]:
df.info()

# Préparer X et y

In [ ]:
y = df["isFraud"]

In [ ]:
X = df.drop(
    columns=["isFraud"]
)

In [ ]:
print(X.shape)
print(y.shape)

### Séparation Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

# Gestion du déséquilibre des classes

In [ ]:
y_train.value_counts()

In [ ]:
y.value_counts(normalize=True) * 100

# Appliquer SMOTE

In [ ]:
pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:
y_train_res.value_counts()

# Premier modèle (Baseline)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_res,
    y_train_res
)

# Prédictions

In [ ]:
y_pred = rf.predict(X_test)

# Évaluation

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

# Matrice de confusion

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

cm

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.show()

# Importance des variables

In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
})

In [ ]:
importance.sort_values(
    by="importance",
    ascending=False
).head(15)

In [ ]:
top = (
    importance
    .sort_values("importance", ascending=False)
    .head(15)
)

plt.figure(figsize=(10,6))

sns.barplot(
    data=top,
    x="importance",
    y="feature"
)

plt.title("Top 15 variables importantes")
plt.show()